# Chapter 6: Advanced SPARQL

**Level 2: Modeling & Querying** · ⏱ about 3 hours · Dataset: *Recipes & Nutrition* + *ratings*

### What you will learn

1. **Aggregates**: `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`, `GROUP_CONCAT`, `SAMPLE`, with `GROUP BY` and `HAVING`
2. **BIND** and expressions: computed values, `IF`, date functions
3. **VALUES**: inline data tables
4. **Subqueries**: "best per group" and "above average"
5. **Property paths**: `/ ^ | + * ?` for walking hierarchies and relationships at any depth
6. **Stardog PATHS** queries: find how two things are connected
7. **Named graph** queries with `GRAPH`
8. **Parameterized queries**: why values must never be pasted into query text
9. **Hands-on:** "all ancestor categories of an ingredient, at any depth", plus new reusable `.rq` queries

In [1]:
import sys
from pathlib import Path

SRC = str(Path('..', 'src').resolve())
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import pandas as pd
import stardog
from rdflib import URIRef

from kg import client, recipes, sparql

pd.set_option('display.max_colwidth', 70)
conn = client.connect()
PREFIX = 'urn:tutorial:ch06'
RECIPES, RATINGS = f'{PREFIX}:recipes', f'{PREFIX}:ratings'
DATA = Path('..', 'data', 'recipes')

with client.transaction(conn) as stats:
    for graph, file in [(RECIPES, 'recipes.ttl'), (RATINGS, 'ratings.nt.gz')]:
        conn.clear(graph_uri=graph)
        conn.add(stardog.content.File(str(DATA / file)), graph_uri=graph)   # .gz is detected and unpacked
print('Loaded:', stats)

PREFIXES = '''PREFIX rec: <http://example.org/recipes#>
PREFIX recipe: <http://example.org/recipes/recipe/>
PREFIX ingredient: <http://example.org/recipes/ingredient/>
PREFIX category: <http://example.org/recipes/category/>
PREFIX allergen: <http://example.org/recipes/allergen/>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
'''
BOTH = [RECIPES, RATINGS]

def q(query, graphs=BOTH, **params):
    return sparql.run_query(conn, PREFIXES + query, graphs=graphs, **params)

Loaded: {'added': 52020, 'removed': 0}


The recipes and the 10,000 ratings from Chapter 5 are in **two graphs**. `q()` queries both merged by default.

## 6.1 Aggregates

An aggregate turns **many rows into one value**. `GROUP BY` decides which rows go together; without it the whole result is one group.

| Function | Result |
|---|---|
| `COUNT(?x)`, `COUNT(DISTINCT ?x)`, `COUNT(*)` | Number of values / distinct values / rows |
| `SUM`, `AVG`, `MIN`, `MAX` | As in SQL |
| `GROUP_CONCAT(?x; separator=", ")` | All values joined into one string |
| `SAMPLE(?x)` | Any one value from the group |

### Whole dataset

In [2]:
q('''SELECT (COUNT(?rating) AS ?ratings) (COUNT(DISTINCT ?user) AS ?users)
         (AVG(?score) AS ?avg) (MIN(?day) AS ?first) (MAX(?day) AS ?last)
WHERE { ?rating a rec:Rating ; rec:byUser ?user ; rec:score ?score ; rec:ratedOn ?day }''')

,ratings,users,avg,first,last
0,10000,1000,3.1103,2025-01-01,2025-12-31


### GROUP BY

How are the scores distributed?

In [3]:
q('SELECT ?score (COUNT(*) AS ?ratings) { ?r rec:score ?score } GROUP BY ?score ORDER BY ?score')

,score,ratings
0,1,669
1,2,2260
2,3,3497
3,4,2447
4,5,1127


Every variable in `SELECT` must either be in `GROUP BY` or inside an aggregate. Group per recipe, and list its cuisine with `SAMPLE`, since each recipe has only one:

In [4]:
q('''SELECT ?recipe (SAMPLE(?c) AS ?cuisine) (COUNT(?rating) AS ?ratings) (ROUND(AVG(?score) * 100) / 100 AS ?avg_score)
WHERE {
  ?rating rec:ofRecipe ?r ; rec:score ?score .
  ?r rec:name ?recipe ; rec:cuisine ?c .
  FILTER(lang(?recipe) = "en")
}
GROUP BY ?recipe
ORDER BY DESC(?avg_score)''')

,recipe,cuisine,ratings,avg_score
0,Margherita Pizza,Italian,736,4.23
1,Grilled Salmon,Mediterranean,714,3.93
2,Lupin Flatbread,Mediterranean,727,3.83
3,Pesto Pasta,Italian,714,3.74
4,Spaghetti Carbonara,Italian,696,3.44
5,Pad Thai,Thai,719,3.30
6,Chocolate Brownies,American,699,2.97
7,Crispy Chicken Tenders,American,715,2.84
8,Potato Salad,German,692,2.80
9,Vegetable Soup,French,724,2.77


### HAVING: filter groups

`FILTER` works on rows **before** grouping; `HAVING` works on groups **after**. Which cuisines average at least 3.5 stars?

In [5]:
q('''SELECT ?cuisine (COUNT(?rating) AS ?ratings) (ROUND(AVG(?score) * 100) / 100 AS ?avg_score)
WHERE { ?rating rec:ofRecipe ?r ; rec:score ?score . ?r rec:cuisine ?cuisine }
GROUP BY ?cuisine
HAVING (AVG(?score) >= 3.5)
ORDER BY DESC(?avg_score)''')

,cuisine,ratings,avg_score
0,Mediterranean,1441,3.88
1,Italian,2146,3.81


### Counting zero

A plain pattern drops things with nothing to count. Combine `OPTIONAL` with `COUNT(?x)`, which ignores unbound values, to get zeros. Which ingredients are used least?

In [6]:
q('''SELECT ?ingredient (COUNT(?r) AS ?used_in)
WHERE {
  ?i a rec:Ingredient ; rec:name ?ingredient .
  OPTIONAL { ?r rec:hasIngredient ?i }
}
GROUP BY ?ingredient
ORDER BY ?used_in ?ingredient
LIMIT 6''', graphs=RECIPES)

,ingredient,used_in
0,Almonds,1
1,Bread Crumbs,1
2,Cocoa Powder,1
3,Coconut Milk,1
4,Green Curry Paste,1
5,Heavy Cream,1


### GROUP_CONCAT

Lists in one cell, e.g. each recipe's allergens:

In [7]:
q('''SELECT ?recipe (GROUP_CONCAT(DISTINCT ?allergen; separator=", ") AS ?allergens)
WHERE {
  ?r a rec:Recipe ; rec:name ?recipe  FILTER(lang(?recipe) = "en")
  OPTIONAL { ?r rec:hasIngredient/rec:hasAllergen/rec:name ?allergen }
}
GROUP BY ?recipe
ORDER BY ?recipe''', graphs=RECIPES)

,recipe,allergens
0,Chicken Stir Fry,"Soybeans, Sesame, Gluten"
1,Chocolate Brownies,"Milk, Eggs, Gluten, Tree Nuts"
2,Crispy Chicken Tenders,"Eggs, Gluten"
3,Grilled Salmon,Fish
4,Lupin Flatbread,Lupin
5,Margherita Pizza,"Gluten, Milk"
6,Moules Marinieres,"Milk, Sulphites, Molluscs"
7,Pad Thai,"Peanuts, Eggs, Fish, Crustaceans, Soybeans"
8,Pancakes,"Milk, Gluten, Eggs"
9,Pesto Pasta,"Gluten, Tree Nuts, Milk"


> 🔁 **Neo4j equivalent:** Cypher groups implicitly by the non-aggregated columns: `RETURN r.cuisine, avg(rt.score)`. `HAVING` becomes `WITH … WHERE`, and `GROUP_CONCAT` is `collect()`.

## 6.2 BIND and expressions

`BIND(expression AS ?var)` creates a new variable **inside** the pattern, so later patterns and filters can use it.

**Energy per serving**, from the ingredient lines measured in grams:

In [8]:
q('''SELECT ?recipe ?servings (ROUND(SUM(?kcal) / ?servings) AS ?kcal_per_serving)
WHERE {
  ?r rec:name ?recipe ; rec:servings ?servings ; rec:hasIngredientLine ?line .
  FILTER(lang(?recipe) = "en")
  ?line rec:unit "g" ; rec:quantity ?grams ; rec:ingredient/rec:kcalPer100g ?kcal100 .
  BIND(?grams * ?kcal100 / 100 AS ?kcal)
}
GROUP BY ?recipe ?servings
ORDER BY DESC(?kcal_per_serving)
LIMIT 5''', graphs=RECIPES)

,recipe,servings,kcal_per_serving
0,Spaghetti Carbonara,2,708.0
1,Margherita Pizza,2,648.0
2,Pesto Pasta,2,563.0
3,Moules Marinieres,2,538.0
4,Crispy Chicken Tenders,3,467.0


This only counts lines in grams. Pieces, millilitres and pinches are skipped, so it's an estimate. Real data is rarely tidy; knowing what a query **leaves out** is part of the job.

### Conditions and dates

`IF(condition, then, else)`, and date functions such as `YEAR`, `MONTH` and `DAY`. Ratings per quarter, labelled with `IF` and `CONCAT`:

In [9]:
q('''SELECT ?quarter (COUNT(*) AS ?ratings) (ROUND(AVG(?score) * 100) / 100 AS ?avg_score)
WHERE {
  ?rating rec:ratedOn ?day ; rec:score ?score .
  BIND(CONCAT("Q", STR(xsd:integer((MONTH(?day) - 1) / 3) + 1)) AS ?quarter)
}
GROUP BY ?quarter ORDER BY ?quarter''', graphs=RATINGS)

,quarter,ratings,avg_score
0,Q1,2549,3.09
1,Q2,2441,3.15
2,Q3,2536,3.10
3,Q4,2474,3.10


In [10]:
q('''SELECT ?recipe ?minutes ?speed
WHERE {
  ?r rec:name ?recipe ; rec:prepMinutes ?minutes  FILTER(lang(?recipe) = "en")
  BIND(IF(?minutes <= 20, "quick", IF(?minutes <= 30, "normal", "slow")) AS ?speed)
}
ORDER BY ?minutes''', graphs=RECIPES).groupby('speed').recipe.apply(', '.join).to_frame()

,recipe
speed,
normal,"Moules Marinieres, Pad Thai, Chicken Stir Fry, Potato Salad, Crisp..."
quick,"Pesto Pasta, Lupin Flatbread, Grilled Salmon, Pancakes, Spaghetti ..."
slow,"Thai Green Curry, Vegetable Soup, Chocolate Brownies"


> ⚠️ `BIND` must come **after** the patterns that set its input variables. A `BIND` placed before them sees unbound values.

## 6.3 VALUES: inline data

`VALUES` puts a small table **inside** the query. The query then runs once for each row. It's useful for lists of things you care about, and for mapping codes to labels.

### A list of values

In [11]:
q('''SELECT ?recipe ?minutes
WHERE {
  VALUES ?r { recipe:pad-thai recipe:pesto-pasta recipe:pancakes }
  ?r rec:name ?recipe ; rec:prepMinutes ?minutes  FILTER(lang(?recipe) = "en")
}''', graphs=RECIPES)

,recipe,minutes
0,Pesto Pasta,15
1,Pad Thai,25
2,Pancakes,20


### A table with several columns

Map each cuisine to a region. The mapping isn't in the data; `VALUES` supplies it for this one query:

In [12]:
q('''SELECT ?region (COUNT(DISTINCT ?r) AS ?recipes) (ROUND(AVG(?score) * 100) / 100 AS ?avg_score)
WHERE {
  VALUES (?cuisine ?region) {
    ("Italian" "Europe") ("French" "Europe") ("German" "Europe") ("Mediterranean" "Europe")
    ("Thai" "Asia") ("Chinese" "Asia")
    ("American" "Americas")
  }
  ?r rec:cuisine ?cuisine .
  ?rating rec:ofRecipe ?r ; rec:score ?score .
}
GROUP BY ?region ORDER BY DESC(?avg_score)''')

,region,recipes,avg_score
0,Europe,8,3.42
1,Americas,3,2.72
2,Asia,3,2.68


`VALUES` is also how you supply parameters when running a `.rq` file by hand in **Studio**: add `VALUES ?max_minutes { 20 }` at the end.

## 6.4 Subqueries

A subquery is a `SELECT` inside a `WHERE`. It's evaluated **first**, and its result joins with the rest like any other pattern. You need one whenever you must aggregate **and then** use the aggregate on a row level.

### Above average

In [13]:
q('''SELECT ?recipe ?avg_score ?overall
WHERE {
  { SELECT (AVG(?s) AS ?overall_raw) WHERE { ?x rec:score ?s } }                    # one row: the global average
  { SELECT ?r (AVG(?s) AS ?avg) WHERE { ?x rec:ofRecipe ?r ; rec:score ?s } GROUP BY ?r }
  FILTER(?avg > ?overall_raw)
  ?r rec:name ?recipe  FILTER(lang(?recipe) = "en")
  BIND(ROUND(?avg * 100) / 100 AS ?avg_score)
  BIND(ROUND(?overall_raw * 100) / 100 AS ?overall)
}
ORDER BY DESC(?avg_score)''')

,recipe,avg_score,overall
0,Margherita Pizza,4.23,3.11
1,Grilled Salmon,3.93,3.11
2,Lupin Flatbread,3.83,3.11
3,Pesto Pasta,3.74,3.11
4,Spaghetti Carbonara,3.44,3.11
5,Pad Thai,3.30,3.11


### Best per group

"The best-rated recipe **of each cuisine**" needs two aggregates: the average per recipe, and the maximum of those averages per cuisine. The query is in `src/kg/queries/recipes/top_rated_by_cuisine.rq`:

In [14]:
print(sparql.load_query('recipes/top_rated_by_cuisine'))
sparql.run_query(conn, 'recipes/top_rated_by_cuisine', graphs=BOTH)

# The best-rated recipe of each cuisine (needs the recipes and ratings graphs).
# Parameters: none
PREFIX rec: <http://example.org/recipes#>

SELECT ?cuisine ?recipe_name ?avg_score ?ratings
WHERE {
  # 1. average score per recipe
  {
    SELECT ?recipe (AVG(?score) AS ?avg) (COUNT(?rating) AS ?ratings)
    WHERE { ?rating rec:ofRecipe ?recipe ; rec:score ?score }
    GROUP BY ?recipe
  }
  ?recipe rec:cuisine ?cuisine ; rec:name ?recipe_name .
  FILTER(lang(?recipe_name) = "en")
  # 2. the best average within each cuisine
  {
    SELECT ?cuisine (MAX(?recipe_avg) AS ?best)
    WHERE {
      { SELECT ?r (AVG(?s) AS ?recipe_avg) WHERE { ?x rec:ofRecipe ?r ; rec:score ?s } GROUP BY ?r }
      ?r rec:cuisine ?cuisine .
    }
    GROUP BY ?cuisine
  }
  # 3. keep the recipe whose average is the best. Two separately computed averages can
  #    differ in the last digits, so compare with a tolerance instead of "="
  FILTER(ABS(?avg - ?best) < 0.000001)
  BIND(ROUND(?avg * 100) / 100 AS ?avg_

,cuisine,recipe_name,avg_score,ratings
0,Italian,Margherita Pizza,4.23,736
1,Mediterranean,Grilled Salmon,3.93,714
2,Thai,Pad Thai,3.30,719
3,American,Chocolate Brownies,2.97,699
4,German,Potato Salad,2.80,692
5,French,Vegetable Soup,2.77,724
6,Chinese,Chicken Stir Fry,2.35,716


Look at step 3: `FILTER(ABS(?avg - ?best) < 0.000001)`. The first version of this query used `FILTER(?avg = ?best)` and returned **no rows at all**. The two averages are computed separately, and decimal division can differ in the last digits. When you compare **computed** decimals, use a tolerance.

> 🔁 **Neo4j equivalent:** `CALL { … }` subqueries, or `WITH` chains: `MATCH … WITH r, avg(s) AS a ORDER BY a DESC WITH r.cuisine AS c, collect(r)[0] AS best`.

## 6.5 Property paths

Property paths describe **routes** through the graph instead of single steps:

| Path | Meaning | Example |
|---|---|---|
| `a/b` | `a` then `b` | `rec:hasIngredient/rec:name` |
| `^a` | `a` backwards | `^rec:hasIngredient`: from an ingredient to its recipes |
| `a\|b` | `a` or `b` | `rec:name\|rdfs:label` |
| `a+` | one or more `a` | `skos:broader+`: all ancestors |
| `a*` | zero or more `a` | `skos:broader*`: itself and all ancestors |
| `a?` | zero or one `a` | `skos:broader?`: itself or its parent |
| `!a` | any property except `a` | `!rdf:type` |

### Walking the category hierarchy

Chapter 3 gave every ingredient a category, and categories a parent (`skos:broader`):

In [15]:
q('''SELECT ?category ?parent ?grandparent
WHERE {
  ?c a rec:Category ; rec:name ?category .
  OPTIONAL { ?c skos:broader/rec:name ?parent }
  OPTIONAL { ?c skos:broader/skos:broader/rec:name ?grandparent }
}
ORDER BY ?grandparent ?parent ?category''', graphs=RECIPES)

,category,parent,grandparent
0,Food,NaN,NaN
1,Dairy,Food,NaN
2,Grains & Cereals,Food,NaN
3,Pantry,Food,NaN
4,Produce,Food,NaN
5,Protein,Food,NaN
6,Cheese,Dairy,Food
7,Milk & Cream,Dairy,Food
8,Bread & Crumbs,Grains & Cereals,Food
9,Flour,Grains & Cereals,Food


Hard-coding `broader/broader` only goes two levels. With `+` and `*` the depth doesn't matter. **All ingredients that are some kind of Dairy**, at any depth:

In [16]:
q('''SELECT ?ingredient ?direct_category
WHERE {
  ?i rec:name ?ingredient ; rec:inCategory ?c .
  ?c rec:name ?direct_category ;
     skos:broader* category:dairy .          # the category itself, or anything below Dairy
}
ORDER BY ?direct_category ?ingredient''', graphs=RECIPES)

,ingredient,direct_category
0,Mozzarella,Cheese
1,Parmesan Cheese,Cheese
2,Butter,Dairy
3,Heavy Cream,Milk & Cream
4,Milk,Milk & Cream


Butter sits directly in *Dairy*, and Parmesan in *Cheese* under *Dairy*. With `skos:broader+` instead of `*`, Butter would be missing: `+` needs at least one step.

### Backwards and alternatives

`^rec:hasIngredient` walks from an ingredient **back** to its recipes. Recipes that share an ingredient with Pad Thai, via ingredient → recipe:

In [17]:
q('''SELECT ?other (GROUP_CONCAT(DISTINCT ?shared; separator=", ") AS ?shared_ingredients) (COUNT(DISTINCT ?i) AS ?n)
WHERE {
  recipe:pad-thai rec:hasIngredient ?i .
  ?i ^rec:hasIngredient ?o ; rec:name ?shared .
  ?o rec:name ?other  FILTER(?o != recipe:pad-thai && lang(?other) = "en")
}
GROUP BY ?other ORDER BY DESC(?n)''', graphs=RECIPES)

,other,shared_ingredients,n
0,Spaghetti Carbonara,"Garlic, Egg",2
1,Thai Green Curry,"Fish Sauce, Lime",2
2,Moules Marinieres,Garlic,1
3,Grilled Salmon,Garlic,1
4,Pancakes,Egg,1
5,Chicken Stir Fry,Garlic,1
6,Pesto Pasta,Garlic,1
7,Crispy Chicken Tenders,Egg,1
8,Chocolate Brownies,Egg,1


That's the beginning of a **"similar recipes"** feature. Chapter 12 does it properly.

> ⚠️ `+` and `*` without a fixed start (e.g. `?a skos:broader* ?b`) can explore a huge part of the graph. Anchor at least one end when you can. Chapter 15 measures the cost.

> 🔁 **Neo4j equivalent:** `-[:BROADER*0..]->` is `skos:broader*`, `-[:BROADER*1..]->` is `+`, and `<-[:HAS_INGREDIENT]-` is `^rec:hasIngredient`.

## 6.6 Stardog PATHS queries

Property paths tell you **whether** things are connected. Stardog's **PATHS** extension tells you **how**: it returns the actual routes.

How is Tofu connected to Parmesan? Two ingredients are "neighbours" when they appear in the same recipe:

In [18]:
result = conn.paths(PREFIXES + '''
PATHS SHORTEST
  START ?x = ingredient:tofu
  END   ?y = ingredient:parmesan-cheese
  VIA   { ?x ^rec:hasIngredient/rec:hasIngredient ?y }''', default_graph_uri=RECIPES)

paths, current = [], []
for row in result['results']['bindings']:
    if not row:                                  # an empty row separates two paths
        paths.append(current); current = []
    else:
        current.append((row['x']['value'].rsplit('/', 1)[1], row['y']['value'].rsplit('/', 1)[1]))
paths.append(current)
for p in paths:
    print(' -> '.join([p[0][0]] + [step[1] for step in p]))

tofu -> egg -> parmesan-cheese
tofu -> garlic -> parmesan-cheese


- `PATHS SHORTEST` returns all shortest routes; `PATHS ALL` returns every route; `CYCLIC` finds loops.
- `VIA` is either a property (or property path) or a **pattern** linking `?x` to `?y`, as here.
- Each result row is **one edge** of a path, and an empty row separates paths.

Tofu (Pad Thai) → Egg or Garlic → Parmesan (Carbonara or Pesto): two hops, two routes.

## 6.7 Named graph queries

Our data sits in two graphs. `GRAPH ?g { … }` makes the graph a variable, so a query can report **where** data comes from. With `FROM NAMED` it can also pick different data from different graphs:

In [19]:
sparql.run_query(conn, PREFIXES + f'''
SELECT ?graph (COUNT(*) AS ?triples) (COUNT(DISTINCT ?type) AS ?types)
FROM NAMED <{RECIPES}> FROM NAMED <{RATINGS}>
WHERE {{ GRAPH ?graph {{ ?s ?p ?o  OPTIONAL {{ ?s a ?type }} }} }}
GROUP BY ?graph''')

,graph,triples,types
0,urn:tutorial:ch06:recipes,1020,5
1,urn:tutorial:ch06:ratings,51000,2


In [20]:
sparql.run_query(conn, PREFIXES + f'''
SELECT ?recipe (COUNT(?rating) AS ?five_star_ratings)
FROM NAMED <{RECIPES}> FROM NAMED <{RATINGS}>
WHERE {{
  GRAPH <{RATINGS}> {{ ?rating rec:ofRecipe ?r ; rec:score 5 }}     # facts from the ratings graph …
  GRAPH <{RECIPES}> {{ ?r rec:name ?recipe  FILTER(lang(?recipe) = "en") }}   # … joined with the recipes graph
}}
GROUP BY ?recipe ORDER BY DESC(?five_star_ratings) LIMIT 5''')

,recipe,five_star_ratings
0,Margherita Pizza,318
1,Grilled Salmon,203
2,Lupin Flatbread,187
3,Pesto Pasta,148
4,Spaghetti Carbonara,86


Pinning each pattern to a graph matters once graphs **disagree**, e.g. an old and a new version of a dataset, or two sources with different values. Then you choose which graph to trust for which fact.

## 6.8 Parameterized queries

User input (a search box, an API parameter) often has to end up in a query. The tempting way is string formatting:

```python
f'... FILTER(CONTAINS(LCASE(?name), LCASE("{search}"))) ...'
```

It breaks as soon as the input contains a quote:

In [21]:
def search_formatted(search):
    return q(f'''SELECT ?ingredient WHERE {{
      ?i a rec:Ingredient ; rec:name ?ingredient
      FILTER(CONTAINS(LCASE(?ingredient), LCASE("{search}"))) }}''', graphs=RECIPES)

print(search_formatted('oil').ingredient.tolist())
try:
    search_formatted('Mum\'s "secret" sauce')
except stardog.exceptions.StardogException as e:
    print('HTTP', e.http_code, str(e).splitlines()[0][:110])

['Sesame Oil', 'Olive Oil']


HTTP 400 [400] QE0PE2: com.complexible.stardog.plan.eval.ExecutionException: Invalid SPARQL query: Encountered " <FUNCT


The quotes inside the input **ended the string early**, so Stardog saw broken SPARQL. That's a bug for a harmless input. For a carefully crafted input it's worse: text that closes the string can add query syntax of its own, changing what the query reads or, in an update, what it deletes. This is **SPARQL injection**, the same class of attack as SQL injection. The fix is the same too: **keep values out of the query text.**

### The fix: bindings

With bindings, the query text never changes. The value travels separately and is **always** treated as a single literal, whatever characters it contains:

In [22]:
SEARCH = PREFIXES + '''SELECT ?ingredient WHERE {
  ?i a rec:Ingredient ; rec:name ?ingredient
  FILTER(CONTAINS(LCASE(?ingredient), LCASE(?search))) }'''

print(sparql.run_query(conn, SEARCH, graphs=RECIPES, search='oil').ingredient.tolist())
print(sparql.run_query(conn, SEARCH, graphs=RECIPES, search='Mum\'s "secret" sauce').ingredient.tolist(),
      '← no match, and no error')

['Sesame Oil', 'Olive Oil']


[] ← no match, and no error


`run_query(…, search=…)` sends `search` as a binding, converted by `sparql.term()`, so it's safe for any input.

**Rules**

1. **Values** (search terms, IDs, numbers, dates) → always **bindings**.
2. **Structure** (graph names, property names, sort order) can't be bound. Choose them from a **fixed list** in code, never from raw input.
3. If you ever must put a value into text, escape it with `sparql.term(value)`, which produces a correctly quoted RDF term.

In [23]:
for value in ['oil', 'Mum\'s "secret" sauce', 'line\nbreak']:
    print(f'{value!r:28} -> {sparql.term(value)}')

'oil'                        -> "oil"
'Mum\'s "secret" sauce'      -> "Mum's \"secret\" sauce"
'line\nbreak'                -> """line
break"""


> 🔁 **Neo4j equivalent:** Cypher parameters (`$search`) do the same job. The rule "never concatenate user input into a query" is identical.

## 6.9 Hands-on: all ancestor categories, at any depth

The task: given an ingredient, list **every** category above it, with how many **levels up** each one is.

`skos:broader*` finds the ancestors. For the level, count the categories passed on the way: every `?step` that lies between the ingredient's category and the ancestor.

In [24]:
print(sparql.load_query('recipes/ingredient_ancestors'))
for name in ['parmesan-cheese', 'chicken-breast', 'butter']:
    ancestors = sparql.run_query(conn, 'recipes/ingredient_ancestors', graphs=RECIPES,
                                 ingredient=recipes.INGREDIENT[name])
    print(f'{name:16}', ' → '.join(f'{c} ({n})' for c, n in zip(ancestors.category, ancestors.levels_up)))

# All categories above an ingredient, at any depth, with the number of steps up.
# Parameters: ?ingredient (IRI, e.g. ingredient:parmesan-cheese)
PREFIX rec:  <http://example.org/recipes#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT ?category (COUNT(?step) AS ?levels_up)
WHERE {
  ?ingredient rec:inCategory ?direct .
  ?direct skos:broader* ?ancestor .
  ?ancestor rec:name ?category .
  # every category passed on the way from ?direct up to ?ancestor counts as one level
  ?direct skos:broader* ?step .
  ?step skos:broader* ?ancestor .
}
GROUP BY ?category
ORDER BY ?levels_up



parmesan-cheese  Cheese (1) → Dairy (2) → Food (3)


chicken-breast   Poultry (1) → Meat (2) → Protein (3) → Food (4)


butter           Dairy (1) → Food (2)


Chicken sits four levels deep (Poultry → Meat → Protein → Food), Butter only two. The query doesn't care about depth.

The same idea, reversed, answers "recipes with **any** dairy", and works for any category at any level:

In [25]:
for category in ['dairy', 'seafood', 'nuts-seeds']:
    print(f'--- {category}')
    display(sparql.run_query(conn, 'recipes/recipes_in_category', graphs=RECIPES, category=recipes.CATEGORY[category]))

--- dairy


,recipe_name,because_of
0,Chocolate Brownies,Butter
1,Margherita Pizza,Mozzarella
2,Moules Marinieres,"Heavy Cream, Butter"
3,Pancakes,"Milk, Butter"
4,Pesto Pasta,Parmesan Cheese
5,Spaghetti Carbonara,Parmesan Cheese


--- seafood


,recipe_name,because_of
0,Grilled Salmon,Salmon
1,Moules Marinieres,Mussels
2,Pad Thai,Shrimp


--- nuts-seeds


,recipe_name,because_of
0,Chocolate Brownies,Almonds
1,Pad Thai,Peanuts
2,Pesto Pasta,Pine Nuts


And the nutrition query from 6.2, now a reusable file:

In [26]:
sparql.run_query(conn, 'recipes/recipe_nutrition', graphs=RECIPES)

,recipe_name,servings,kcal_per_serving,protein_g_per_serving,lines_counted
0,Moules Marinieres,2,538.0,60.0,2
1,Crispy Chicken Tenders,3,467.0,58.0,3
2,Chicken Stir Fry,3,371.0,44.0,3
3,Thai Green Curry,4,319.0,41.0,3
4,Margherita Pizza,2,648.0,31.0,3
5,Spaghetti Carbonara,2,708.0,30.0,3
6,Grilled Salmon,2,312.0,30.0,1
7,Pad Thai,2,306.0,28.0,4
8,Pesto Pasta,2,563.0,23.0,4
9,Lupin Flatbread,4,186.0,20.0,1


The query catalogue has grown:

In [27]:
sparql.list_queries('recipes')

,query,description,parameters
0,recipes/all_recipes,"All recipes with cuisine, prep time and servings, quickest first.",none
1,recipes/french_menu,"The menu in French, with the English description where there is one.",none
2,recipes/high_protein_ingredients,Ingredients with at least ?min_protein grams of protein per 100 g.,?min_protein (number)
3,recipes/ingredient_ancestors,"All categories above an ingredient, at any depth, with the number ...","?ingredient (IRI, e.g. ingredient:parmesan-cheese)"
4,recipes/meat_free_recipes,"Recipes without meat, poultry, pork, fish or shellfish.",none
5,recipes/quick_recipes,Recipes ready in at most ?max_minutes minutes.,?max_minutes (integer)
6,recipes/recipe_allergens,"Every recipe with each allergen it contains, and the ingredient re...",none
7,recipes/recipe_nutrition,"Energy and protein per serving, from ingredient lines measured in ...",none
8,recipes/recipes_by_cuisine,Recipes of one cuisine.,"?cuisine (text, e.g. ""Italian"")"
9,recipes/recipes_in_category,Recipes with at least one ingredient in a category or any of its s...,"?category (IRI, e.g. category:dairy)"


## 6.10 Exercises

**Exercise 1.** For each **user**, count their ratings and their average score. Show only users with **at least 15 ratings**, best average first, top 5.

<details><summary>Solution</summary>

```python
q('''SELECT ?user (COUNT(?r) AS ?ratings) (AVG(?s) AS ?avg)
WHERE { ?r rec:byUser ?user ; rec:score ?s }
GROUP BY ?user HAVING (COUNT(?r) >= 15)
ORDER BY DESC(?avg) LIMIT 5''', graphs=RATINGS)
```
</details>

**Exercise 2.** Using a property path, list every **category below Protein** (at any depth) with its direct parent.

<details><summary>Solution</summary>

```python
q('''SELECT ?category ?parent WHERE {
  ?c skos:broader+ category:protein ; rec:name ?category ; skos:broader/rec:name ?parent }
ORDER BY ?parent ?category''', graphs=RECIPES)
```
</details>

**Exercise 3.** Use `VALUES` to rate **two** recipes side by side: count their 1-star and 5-star ratings.

<details><summary>Solution</summary>

```python
q('''SELECT ?r (SUM(IF(?s = 1, 1, 0)) AS ?one_star) (SUM(IF(?s = 5, 1, 0)) AS ?five_star)
WHERE { VALUES ?r { recipe:margherita-pizza recipe:chicken-stir-fry }
        ?x rec:ofRecipe ?r ; rec:score ?s }
GROUP BY ?r''', graphs=RATINGS)
```
</details>

**Exercise 4.** Write `recipes_containing_allergen.rq`: recipes containing a given allergen, with a **parameter** `?allergen` (an IRI). Run it with `allergen=recipes.ALLERGEN['eggs']` via `run_query`.

<details><summary>Solution</summary>

```sparql
# Recipes containing a given allergen, and the ingredients responsible.
# Parameters: ?allergen (IRI, e.g. allergen:eggs)
PREFIX rec: <http://example.org/recipes#>
SELECT ?recipe_name (GROUP_CONCAT(?ingredient; separator=", ") AS ?because_of)
WHERE {
  ?r rec:name ?recipe_name ; rec:hasIngredient ?i .
  ?i rec:hasAllergen ?allergen ; rec:name ?ingredient .
  FILTER(lang(?recipe_name) = "en")
}
GROUP BY ?recipe_name ORDER BY ?recipe_name
```
```python
sparql.run_query(conn, 'recipes/recipes_containing_allergen', graphs=RECIPES, allergen=recipes.ALLERGEN['eggs'])
```
</details>

In [28]:
# your answers here

## 6.11 Clean up

In [29]:
CLEAN_UP = True

if CLEAN_UP:
    with client.transaction(conn):
        for g in (RECIPES, RATINGS):
            conn.clear(graph_uri=g)
    print('Removed', RECIPES, 'and', RATINGS)
conn.close()

Removed urn:tutorial:ch06:recipes and urn:tutorial:ch06:ratings


## Summary

- **Aggregates** summarize groups: `GROUP BY` forms the groups, `HAVING` filters them, and `OPTIONAL` + `COUNT` gives zeros.
- **BIND** computes new values inside the pattern; `IF`, `CONCAT`, `ROUND` and date functions help.
- **VALUES** adds an inline table: lists of values, code-to-label mappings, parameters in Studio.
- **Subqueries** run first. Use them to aggregate and then use the result per row ("above average", "best per group"). Compare **computed decimals with a tolerance**, not `=`.
- **Property paths** (`/ ^ | + * ?`) walk relationships and hierarchies at any depth. Anchor unbounded paths.
- **Stardog PATHS** returns the actual routes between two things.
- **GRAPH ?g** with `FROM NAMED` reports and controls which graph each fact comes from.
- **Never paste values into query text.** Use **bindings** (`run_query(…, name=value)`); choose structure from fixed lists.

**Next: Chapter 7, Ontology design.** Turn the recipes vocabulary into an RDFS/OWL ontology with classes, properties, domains and ranges, and register it in Stardog as a named model.